# REFIT Electrical Load Measurements: Data Cleaning and Appliance Disaggregation
**End-to-End Pipeline for Flock Energy Take-Home Assignment**

This notebook provides a complete, reproducible execution flow covering:
1. **Section 1**: Basic Inspection and Cleaning of Raw Data (House 1)
2. **Section 2**: Exploratory Data Analysis of Washing Machine Usage Across 4 Homes (Houses 1, 2, 3, 20)
3. **Section 3**: Building & Evaluating an Appliance Disaggregation (NILM) Model & LOHO Cross-Validation
4. **Section 4**: Practical Implications & Utility Recommendations

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Set project root
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import HOUSE_METADATA, RAW_DATA_DIR, CLEAN_DATA_DIR, OUTPUT_DIR, FIGURES_DIR
from src.data_inspection import run_inspection
from src.data_cleaning import clean_raw_house1
from src.eda_washing_machine import run_eda
from src.disaggregation_model import main as run_disaggregation

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Environment initialized successfully.")

## 1. Basic Inspection and Cleaning of Raw Data (House 1)
We inspect timestamp monotonicity, sampling distributions, missing gaps, and impossible spikes/sensor codes.

In [ ]:
# Run data inspection
inspection_summary = run_inspection()
for k, v in inspection_summary.items():
    if 'total_rows' in v:
        print(f"{k}: {v['total_rows']:,} rows | Duplicates: {v['duplicate_timestamps']:,} | Outages: {v['gap_breakdown']['outages_gt_180s']}")

In [ ]:
# Execute cleaning pipeline: sorting, deduplication, 1-min resampling, and outage flagging
cleaning_results = clean_raw_house1()
print("Cleaning results:", cleaning_results)

### Before vs. After Cleaning Plot
Below is the comparison of raw irregular readings vs cleaned 1-minute uniform data.

In [ ]:
from IPython.display import Image
Image(filename=str(FIGURES_DIR / "sec1_raw_vs_cleaned.png"))

## 2. Exploratory Analysis of Washing Machine Usage Across 4 Homes
We evaluate Houses 1, 2, 3, and 20 at 1-minute resolution, identifying active wash cycles using a multi-phase soak-bridging rule.

In [ ]:
eda_summary, house_dfs, house_cycles = run_eda()
df_summary = pd.DataFrame(eda_summary).T
df_summary[['occupancy', 'detected_cycles', 'cycles_per_week', 'zero_off_pct', 'mean_duration_mins', 'mean_energy_kwh']]

### Plot (i): 24-Hour Whole-Home vs. Washing Machine Trace (Active Day 2015-02-23)

In [ ]:
Image(filename=str(FIGURES_DIR / "sec2_24h_trace.png"))

### Plot (ii): Zoomed-In Detected Washing Machine Cycle (Pulsed Heating Bursts)

In [ ]:
Image(filename=str(FIGURES_DIR / "sec2_detected_cycles.png"))

### Plot (iii): Cross-House Characteristic Comparison

In [ ]:
Image(filename=str(FIGURES_DIR / "sec2_cross_house_comparison.png"))

### Plot (iv): Temporal Usage Patterns (Diurnal & Day-of-Week)

In [ ]:
Image(filename=str(FIGURES_DIR / "sec2_temporal_patterns.png"))

## 3. Disaggregation Model Training, Evaluation & LOHO Cross-Validation
We train an offline Gradient Boosting model with temporal window features on aggregate power and evaluate against baselines and across unseen homes.

In [ ]:
run_disaggregation()

### Disaggregation Evaluation: Actual vs. Predicted Plot

In [ ]:
Image(filename=str(FIGURES_DIR / "sec3_actual_vs_predicted.png"))

### Disaggregation Evaluation: On/Off Confusion Matrix

In [ ]:
Image(filename=str(FIGURES_DIR / "sec3_confusion_matrix.png"))

### Disaggregation Evaluation: Cumulative Energy Accounting

In [ ]:
Image(filename=str(FIGURES_DIR / "sec3_cumulative_energy.png"))